# CourtView — tracking pipeline (Google Colab)

Runs player detection, tracking and court mapping on the demo clips and gives you a zip of
`telemetry.json` / `roster.json` files to drop into `web/public/clips/`.

**Before you start**

1. `Runtime → Change runtime type → T4 GPU`.
2. Create a free Roboflow account, copy your API key (Settings → API Keys).
3. In Colab open the key icon (Secrets) in the left bar, add a secret named `ROBOFLOW_API_KEY`
   with the key as its value, and enable notebook access. Do not paste the key into a cell.

In [ ]:
!nvidia-smi -L

import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[CUDAExecutionProvider]"

## 1. Get the project

Clones the repository (pipeline code + the demo clips in `web/public/clips/`). Push your latest
local changes first. For a private repository, upload a zip of the project folder with the file
browser instead and unzip it to `/content/basketball-stream-tracker`.

In [ ]:
REPO_URL = "https://github.com/realcyclopse/basketball-stream-tracker.git"

%cd /content
![ -d basketball-stream-tracker ] || git clone --depth 1 $REPO_URL basketball-stream-tracker
!ls basketball-stream-tracker/web/public/clips

## 2. Install dependencies

In [ ]:
!pip install -q inference-gpu supervision==0.27.0 scipy tqdm "git+https://github.com/roboflow/sports.git@feat/basketball"

## 3. (Optional) SAM2 real-time tracker

Gives pixel-accurate player outlines and steadier ids than the default ByteTrack box tracker.
Takes a few minutes to install. Skip this cell to use ByteTrack; the pipeline picks whichever
is available.

In [ ]:
%cd /content
![ -d segment-anything-2-real-time ] || git clone https://github.com/Gy920/segment-anything-2-real-time.git
%cd /content/segment-anything-2-real-time
!pip install -q -e .
!python setup.py build_ext --inplace
![ -f checkpoints/sam2.1_hiera_large.pt ] || (cd checkpoints && bash download_ckpts.sh)

os.environ["SAM2_CHECKPOINT"] = "/content/segment-anything-2-real-time/checkpoints/sam2.1_hiera_large.pt"
%cd /content

## 4. Self-test (no API key needed)

Runs the pipeline on a generated clip with stand-in detectors. Every line should say `PASS`.

In [ ]:
%cd /content/basketball-stream-tracker/pipeline
!python test_pipeline.py

## 5. Run on every demo clip

Processes each `web/public/clips/<clip>/video.mp4` and writes `telemetry.json`, a draft
`roster.json` and `frame_0_calibration.jpg` next to it.

In [ ]:
%cd /content/basketball-stream-tracker/pipeline
!python extract_telemetry.py --tracker auto

## 6. Check the calibration frames

Each tracked player is labelled `ID: <n>`. Use these to fill in names, jersey numbers and teams
in each clip's `roster.json` (the pipeline only guesses the team from jersey colour).

In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob("/content/basketball-stream-tracker/web/public/clips/*/frame_0_calibration.jpg")):
    print(path)
    display(Image(path, width=960))

## 7. Download the results

Unzip `tracking_outputs.zip` into `web/public/clips/` of your local project (it contains one
folder per clip), then reload the web app.

In [ ]:
%cd /content/basketball-stream-tracker/web/public/clips
!rm -f /content/tracking_outputs.zip
!zip -q /content/tracking_outputs.zip */telemetry.json */roster.json */frame_0_calibration.jpg -x "synthetic/*"
!unzip -l /content/tracking_outputs.zip

from google.colab import files
files.download("/content/tracking_outputs.zip")